# Exploratory Data Analysis of the Ames Housing Dataset
**Dataset:** Kaggle *House Prices: Advanced Regression Techniques* (`train.csv`, 1,460 rows x 81 columns)  
**Objective:** use EDA only (no machine learning) to understand which factors drive sale price.

## 1. Introduction - guiding questions
1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does `OverallQual` change price in a clear, ordered way?
4. Do some neighborhoods consistently cost more than others?
5. Are there outliers, and are they errors or real but unusual sales?
6. Which columns have missing values, and which of those are not truly missing?

*Place `train.csv` in the same folder as this notebook and run the cells in order.*

## 2. Data loading (Step 1)
**What I'm about to do:** import libraries, load the data, and check shape, types and missing values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("train.csv")
print("Shape:", df.shape)
display(df.head())
df.info()

missing = df.isnull().sum().sort_values(ascending=False)
miss_table = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})
miss_table = miss_table[miss_table["missing"] > 0]
print(len(miss_table), "columns have missing values")
display(miss_table)

**What I learned:** 1,460 rows, 81 columns, and 19 columns with missing values. PoolQC (99.5%), MiscFeature, Alley and Fence are almost entirely empty, which hints that a gap often means *feature absent*. Garage columns all have 81 gaps and basement columns 37-38, matching houses with no garage or basement. `MasVnrType` shows 872 gaps because most values are the text "None", which pandas reads as missing.

## 3. Data understanding (Steps 2 and 3)
**What I'm about to do:** classify variables, mark ordinal columns, and compute descriptive statistics.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]
ordinal_map = {"ExterQual": quality_order, "KitchenQual": quality_order}
for col, order in ordinal_map.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

def kind(c):
    if c in ordinal_map or c == "OverallQual":
        return "ordinal"
    return "numerical" if c in numeric_cols else "categorical"

summary = pd.DataFrame({
    "column": df.columns,
    "type": [kind(c) for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "examples": [list(df[c].dropna().unique()[:3]) for c in df.columns],
})
display(summary)

**What I learned:** 38 numeric and 43 text columns. `OverallQual` is stored as a number but is really ordinal (the step from 4 to 5 need not equal 8 to 9), so it is treated as ordinal in the interpretation.

In [ ]:
display(df[numeric_cols].describe().T)

rows = []
for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col]
    q1, q3 = s.quantile([0.25, 0.75])
    note = ("right skewed" if s.mean() > s.median()
            else "left skewed" if s.mean() < s.median() else "symmetric")
    rows.append({"column": col, "mean": s.mean(), "median": s.median(), "mode": s.mode().iloc[0],
                 "std": s.std(), "variance": s.var(), "range": s.max() - s.min(), "iqr": q3 - q1,
                 "skew": s.skew(), "kurtosis": s.kurt(), "shape": note})
display(pd.DataFrame(rows).set_index("column").round(2))

for col in ["Neighborhood", "HouseStyle"]:
    print(df[col].value_counts())
    print("Mode:", df[col].mode()[0], "\n")

**What I learned:** SalePrice mean (about $180,921) is above its median ($163,000), so a few expensive homes pull the mean up: a **right-skewed** distribution (skew 1.88). The median is the better "typical" price. LotArea is extremely skewed (skew 12.2). NAmes is the most common neighborhood (225) and 1Story the most common style (726).

## 4. Data cleaning (Step 9, plus the Step 8 decision)
**What I'm about to do:** handle outliers that are non-typical sales, then check duplicates and labels, and fill missing values with documented rules. (The outlier evidence is shown in Section 7.)

In [ ]:
# Two very large houses sold far below trend; both are 'Partial' sales (see Section 7).
bad = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
print("Removing Ids:", df.loc[bad, "Id"].tolist())
df_clean = df.loc[~bad].copy()

# 1. Duplicates
dup_count = df_clean.duplicated().sum()
print("Duplicate rows:", dup_count)
if dup_count > 0:
    df_clean = df_clean.drop_duplicates()

# 2. Inconsistent labels
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    vals = df_clean[col].dropna().unique()
    if len({str(v).strip().lower() for v in vals}) < len(vals):
        print("Inconsistent labels in", col)
        df_clean[col] = df_clean[col].str.strip()

# 3. NA means 'feature absent'
none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageType",
             "GarageFinish", "GarageQual", "GarageCond", "BsmtQual", "BsmtCond",
             "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"]
for col in none_cols:
    df_clean[col] = df_clean[col].fillna("None")

# 4. Genuinely missing numeric values
df_clean["LotFrontage"] = (df_clean.groupby("Neighborhood")["LotFrontage"]
                           .transform(lambda s: s.fillna(s.median())))
df_clean["LotFrontage"] = df_clean["LotFrontage"].fillna(df_clean["LotFrontage"].median())
df_clean["MasVnrArea"] = df_clean["MasVnrArea"].fillna(0)               # assumption
df_clean["GarageYrBlt"] = df_clean["GarageYrBlt"].fillna(df_clean["YearBuilt"])  # assumption
df_clean["Electrical"] = df_clean["Electrical"].fillna(df_clean["Electrical"].mode()[0])

# 5. Verify and check impossible values
print("Remaining missing values:", int(df_clean.isnull().sum().sum()))
print("Built after sold:", (df_clean["YearBuilt"] > df_clean["YrSold"]).sum())
area_cols = [c for c in df_clean.columns if "SF" in c or "Area" in c]
print("Negative areas:", int((df_clean[area_cols] < 0).sum().sum()))
print("Clean shape:", df_clean.shape)

**What I learned:** no duplicates, no inconsistent labels, no impossible values, and zero missing values remain (1,458 rows). Decisions: 15 columns filled with "None" because NA means the feature is absent; LotFrontage filled with the neighborhood median; MasVnrArea = 0 and GarageYrBlt = YearBuilt are *assumptions* worth stating.

## 5. Univariate analysis (Step 4)
**What I'm about to do:** look at the shape of SalePrice and how it varies by quality and neighborhood.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(df["SalePrice"], kde=True, ax=ax[0]); ax[0].set(title="SalePrice", xlabel="Sale price ($)")
sns.boxplot(y=df["SalePrice"], ax=ax[1]); ax[1].set(title="SalePrice boxplot", ylabel="Sale price ($)")
sns.histplot(np.log(df["SalePrice"]), kde=True, ax=ax[2]); ax[2].set(title="log(SalePrice)", xlabel="log price")
plt.tight_layout(); plt.show()
print("Skew before log:", round(df["SalePrice"].skew(), 3))
print("Skew after log :", round(np.log(df["SalePrice"]).skew(), 3))

plt.figure(figsize=(10, 5))
sns.boxplot(x="OverallQual", y="SalePrice", data=df)
plt.title("SalePrice by OverallQual"); plt.show()

order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(13, 5))
sns.boxplot(x="Neighborhood", y="SalePrice", data=df, order=order)
plt.xticks(rotation=90); plt.title("SalePrice by Neighborhood (sorted by median)"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.countplot(x="OverallQual", data=df, ax=ax[0]); ax[0].set_title("Houses by OverallQual")
sns.countplot(x="HouseStyle", data=df, ax=ax[1]); ax[1].set_title("Houses by HouseStyle")
plt.tight_layout(); plt.show()

**What I learned:** skewness falls from 1.88 to 0.12 after the log, so log(SalePrice) is close to normal. Median price rises at every quality grade ($50,150 at grade 1 to $432,390 at grade 10) and the spread widens at the top (extreme grades have very few houses). NridgHt, NoRidge and StoneBr have clearly the highest medians; MeadowV, IDOTRR and BrDale the lowest.

## 6. Bivariate and multivariate analysis (Steps 5, 6, 7)
**What I'm about to do:** rank features by correlation, compare covariance with correlation, and plot key relationships.

In [ ]:
num = df.select_dtypes(include="number").drop(columns="Id")
cov_matrix, corr_matrix = num.cov(), num.corr()

top_corr = (corr_matrix["SalePrice"].drop("SalePrice").abs()
            .sort_values(ascending=False).head(10))
print(corr_matrix["SalePrice"][top_corr.index].round(3))

for f in top_corr.index[:3]:
    print(f"{f:12s} cov = {cov_matrix.loc[f, 'SalePrice']:>14,.0f}   "
          f"corr = {corr_matrix.loc[f, 'SalePrice']:.3f}")

print("Spearman OverallQual:", round(stats.spearmanr(df["OverallQual"], df["SalePrice"])[0], 3))
print("Spearman ExterQual  :", round(stats.spearmanr(df["ExterQual"].cat.codes, df["SalePrice"])[0], 3))
print("GarageArea vs GarageCars:", round(df["GarageArea"].corr(df["GarageCars"]), 3))
print("TotalBsmtSF vs 1stFlrSF :", round(df["TotalBsmtSF"].corr(df["1stFlrSF"]), 3))

**What I learned:** the strongest correlates are OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), GarageArea (0.62), TotalBsmtSF (0.61), 1stFlrSF (0.61). Covariances are in unrelated units (so cannot be compared); correlations are unit-free. GarageArea and GarageCars correlate at 0.88: they carry nearly the same information, so they are not two independent drivers. Correlation is not causation.

In [ ]:
flag = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

plt.figure(figsize=(8, 5))
plt.scatter(df.loc[~flag, "GrLivArea"], df.loc[~flag, "SalePrice"], alpha=0.5, label="Other houses")
plt.scatter(df.loc[flag, "GrLivArea"], df.loc[flag, "SalePrice"], c="red", s=70, label="Large but low price")
plt.title("GrLivArea vs SalePrice"); plt.xlabel("GrLivArea (sq ft)"); plt.ylabel("Sale price ($)")
plt.legend(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(df["TotalBsmtSF"], df["SalePrice"], alpha=0.5); ax[0].set(title="TotalBsmtSF vs SalePrice", xlabel="TotalBsmtSF", ylabel="Sale price ($)")
ax[1].scatter(df["YearBuilt"], df["SalePrice"], alpha=0.5); ax[1].set(title="YearBuilt vs SalePrice", xlabel="YearBuilt", ylabel="Sale price ($)")
plt.tight_layout(); plt.show()

selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]
sns.pairplot(df[selected + ["CentralAir"]], hue="CentralAir", diag_kind="kde", height=2)
plt.show()

top10 = top_corr.index.tolist()
corr_top = df[top10 + ["SalePrice"]].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr_top, annot=True, fmt=".2f", cmap="coolwarm", center=0, mask=mask, square=True, linewidths=0.5)
plt.title("Top 10 features correlated with SalePrice"); plt.tight_layout(); plt.show()

**What I learned:** price rises roughly linearly with living area and basement area, and newer houses tend to cost more (r = 0.52 with YearBuilt). Two red points (Ids 524 and 1299) are far below the trend. Houses with central air have a much higher median ($168,000 vs $98,000), partly because houses without it are mostly older.

## 7. Outlier analysis (Step 8)
**What I'm about to do:** flag outliers with the IQR rule and the z-score rule, compare, and decide what to remove.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series[(series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)]

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return series[z.abs() > threshold]

flagged = {}
for col in ["GrLivArea", "SalePrice"]:
    a, b = iqr_outliers(df[col]), zscore_outliers(df[col])
    flagged[col] = a.index.union(b.index)
    print(f"{col}: IQR={len(a)}  z-score={len(b)}  overlap={len(a.index.intersection(b.index))}")

fig, ax = plt.subplots(1, 2, figsize=(14, 3))
for i, col in enumerate(["GrLivArea", "SalePrice"]):
    sns.boxplot(x=df[col], ax=ax[i]); ax[i].set_title(col)
plt.tight_layout(); plt.show()

is_flag = df.index.isin(flagged["GrLivArea"].union(flagged["SalePrice"]))
plt.figure(figsize=(8, 5))
plt.scatter(df.loc[~is_flag, "GrLivArea"], df.loc[~is_flag, "SalePrice"], alpha=0.5, label="Normal")
plt.scatter(df.loc[is_flag, "GrLivArea"], df.loc[is_flag, "SalePrice"], c="red", s=22, label="Flagged")
plt.title("Flagged points"); plt.xlabel("GrLivArea"); plt.ylabel("Sale price ($)"); plt.legend(); plt.show()

display(df.loc[flag, ["Id", "GrLivArea", "SalePrice", "SaleCondition", "Neighborhood"]])
print("Large AND expensive (kept):")
display(df.loc[(df["GrLivArea"] > 4000) & (df["SalePrice"] >= 300000), ["Id", "GrLivArea", "SalePrice"]])
print("GrLivArea-SalePrice r before:", round(df["GrLivArea"].corr(df["SalePrice"]), 3),
      "| after removal:", round(df_clean["GrLivArea"].corr(df_clean["SalePrice"]), 3))

**What I learned:** IQR flags more than the z-score rule (GrLivArea 31 vs 16; SalePrice 61 vs 22) and every z-score outlier is also an IQR outlier. The long right tail inflates the mean and standard deviation, pushing the z-score cut-off far out, while quartiles resist the tail. Most flagged points are real, expensive or large homes and are **kept**. Ids 524 and 1299 (over 4,600 sq ft, under $185,000, both *Partial* sales) are non-typical sales and were **removed**; the correlation rises from 0.709 to 0.735. Ids 692 and 1183 are equally large but sold for about $750,000, in line with the trend, so they stay.

## 8. Dashboard (Step 10)
**What I'm about to do:** combine the best visuals into one figure and save it as `eda_dashboard.png`.

In [ ]:
main = "#2A7F8E"
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

avg = df_clean.groupby("YearBuilt")["SalePrice"].mean()
ax[0, 0].plot(avg.index, avg.values, color=main, label="Average price")
ax[0, 0].set(title="Average sale price by year built", xlabel="Year built", ylabel="Mean sale price ($)")
ax[0, 0].legend()

top_n = df_clean.groupby("Neighborhood")["SalePrice"].mean().sort_values().tail(10)
ax[0, 1].barh(top_n.index, top_n.values, color=main)
ax[0, 1].set(title="Top 10 neighbourhoods by mean price", xlabel="Mean sale price ($)", ylabel="Neighbourhood")

sns.histplot(df_clean["SalePrice"], kde=True, color=main, ax=ax[1, 0])
ax[1, 0].axvline(df_clean["SalePrice"].mean(), color="red", linestyle="--", label="Mean")
ax[1, 0].axvline(df_clean["SalePrice"].median(), color="black", label="Median")
ax[1, 0].set(title="SalePrice distribution", xlabel="Sale price ($)", ylabel="Count")
ax[1, 0].legend()

sns.scatterplot(data=df_clean, x="GrLivArea", y="SalePrice", hue="OverallQual",
                palette="viridis", alpha=0.7, ax=ax[1, 1])
ax[1, 1].set(title="Living area vs price, by quality", xlabel="GrLivArea (sq ft)", ylabel="Sale price ($)")

fig.suptitle("House Price EDA Summary", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("eda_dashboard.png", dpi=150)
plt.show()

**What I learned:** price trends upward with year built (noisy in early years with few houses); NoRidge, NridgHt and StoneBr lead by mean price; the mean stays above the median after cleaning, so the skew is genuine; and for a given size, higher-quality houses sit higher.

## 9. Conclusions
- **Distribution:** SalePrice is right skewed (skew 1.88, mean > median); a log transform gives skew 0.12.
- **Drivers:** OverallQual (r = 0.79) and GrLivArea (r = 0.71) are the strongest numeric correlates; garage and basement size follow, but overlap with each other.
- **Quality:** price rises in an ordered way with OverallQual, with wider spread at higher grades.
- **Location:** NridgHt, NoRidge and StoneBr cost clearly more; the top-to-bottom median ratio is about 3.6 to 1.
- **Outliers:** two non-typical Partial sales removed; other large, expensive houses are real and kept.
- **Missing data:** 15 of 19 columns with gaps mean "feature absent" and were filled with "None".

**Limitations:** correlation is not causation; Pearson captures only linear association; imputation of MasVnrArea and GarageYrBlt rests on assumptions; some groups are small (OverallQual 1, 2, 10; Blueste with 2 houses); one city, limited period; no predictive model was built.